# 🎓 University Toolkit: Data Analysis & Statistics
**Topics Covered:** Descriptive Stats | Hypothesis Testing | Regression | Visualisation | Grade Analysis

---

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#444', 'axes.labelcolor': '#cdd6f4',
    'xtick.color': '#cdd6f4', 'ytick.color': '#cdd6f4',
    'text.color': '#cdd6f4', 'grid.color': '#2a2d3e',
    'grid.linestyle': '--', 'font.family': 'monospace',
})
COLORS = ['#89b4fa', '#a6e3a1', '#fab387', '#f38ba8', '#cba6f7', '#94e2d5']
print('Libraries loaded.')

Libraries loaded.


---
## 1 · Generate Student Dataset (120 students)

In [ ]:
n = 120
study_hours  = np.clip(np.random.normal(15, 5, n), 2, 40)
attendance   = np.clip(np.random.normal(78, 12, n), 20, 100)
assignments  = np.clip(np.random.normal(72, 14, n), 20, 100)
noise        = np.random.normal(0, 6, n)
final_exam   = np.clip(0.35*study_hours*1.5 + 0.25*attendance*0.5
                       + 0.3*assignments*0.6 + noise + 10, 0, 100)

def grade_band(s):
    if s >= 70: return 'First'
    elif s >= 60: return 'Upper Second'
    elif s >= 50: return 'Lower Second'
    elif s >= 40: return 'Third'
    else: return 'Fail'

df = pd.DataFrame({
    'student_id': [f'STU{i+1:03d}' for i in range(n)],
    'study_hrs': study_hours.round(1),
    'attendance': attendance.round(1),
    'assignments': assignments.round(1),
    'final_exam': final_exam.round(1),
})
df['grade_band'] = df['final_exam'].apply(grade_band)
print(df.head(8).to_string(index=False))
print(f'\nShape: {df.shape}')

---
## 2 · Descriptive Statistics & Distributions

In [ ]:
cols   = ['study_hrs', 'attendance', 'assignments', 'final_exam']
labels = ['Study Hours/week', 'Attendance %', 'Assignment %', 'Final Exam %']
print(df[cols].describe().round(2).to_string())

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle('Distribution of Student Variables', fontsize=14, fontweight='bold', color='#89b4fa')
for i, (col, lbl) in enumerate(zip(cols, labels)):
    ax = axes[i]
    ax.hist(df[col], bins=18, color=COLORS[i], edgecolor='#0f1117', alpha=0.85)
    mu, sig = df[col].mean(), df[col].std()
    ax.axvline(mu, color='white', lw=1.5, linestyle='--', label=f'μ={mu:.1f}')
    ax.axvline(mu+sig, color='#f38ba8', lw=1, linestyle=':')
    ax.axvline(mu-sig, color='#f38ba8', lw=1, linestyle=':', label=f'σ={sig:.1f}')
    ax.set(xlabel=lbl, ylabel='Count' if i==0 else '')
    ax.legend(fontsize=8); ax.grid(True)
plt.tight_layout()
plt.show()

---
## 3 · Correlation Heatmap

In [ ]:
corr = df[cols].corr().round(3)
print(corr.to_string())

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, label='Pearson r')
tks = ['Study hrs', 'Attendance', 'Assignments', 'Final exam']
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(tks, rotation=30, ha='right')
ax.set_yticklabels(tks)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center',
                color='black', fontsize=11, fontweight='bold')
ax.set_title('Correlation Heatmap', fontweight='bold', color='#cba6f7')
plt.tight_layout()
plt.show()

---
## 4 · Multiple Linear Regression
$$\hat{y} = \beta_0 + \beta_1(\text{study hrs}) + \beta_2(\text{attendance}) + \beta_3(\text{assignments})$$

In [ ]:
X = df[['study_hrs', 'attendance', 'assignments']].values
y = df['final_exam'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

model = LinearRegression().fit(X_train, y_train)
y_pred = model.predict(X_test)
r2   = r2_score(y_test, y_pred)
rmse = np.sqrt(np.mean((y_test - y_pred)**2))

print(f'Intercept β₀ = {model.intercept_:.3f}')
for name, c in zip(['Study hrs','Attendance','Assignments'], model.coef_):
    print(f'  {name:<14} β = {c:.3f}')
print(f'\nR² = {r2:.4f}   RMSE = {rmse:.2f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Linear Regression — Exam Score Prediction', fontsize=13, fontweight='bold', color='#a6e3a1')
axes[0].scatter(y_test, y_pred, color=COLORS[0], alpha=0.75, edgecolors='#0f1117', s=60)
lims = [min(y_test.min(),y_pred.min())-2, max(y_test.max(),y_pred.max())+2]
axes[0].plot(lims, lims, '--', color=COLORS[3], lw=1.5, label='Perfect fit')
axes[0].set(xlabel='Actual', ylabel='Predicted', title=f'Actual vs Predicted  (R²={r2:.3f})')
axes[0].legend(); axes[0].grid(True)
residuals = y_test - y_pred
axes[1].hist(residuals, bins=16, color=COLORS[2], edgecolor='#0f1117', alpha=0.85)
axes[1].axvline(0, color='white', lw=1.5, linestyle='--')
axes[1].set(xlabel='Residuals', ylabel='Count', title='Residual Distribution')
axes[1].grid(True)
plt.tight_layout()
plt.show()

---
## 5 · Hypothesis Test — Does Attendance Matter?
**H₀:** High vs low attendance groups have equal mean exam scores  
**H₁:** High-attendance students score higher *(one-tailed t-test, α = 0.05)*

In [ ]:
threshold = 80
high = df.loc[df['attendance'] >= threshold, 'final_exam'].values
low  = df.loc[df['attendance'] <  threshold, 'final_exam'].values
t_stat, p_two = stats.ttest_ind(high, low)
p_one = p_two / 2

print(f'High-attendance  n={len(high)}  mean={high.mean():.2f}')
print(f'Low-attendance   n={len(low)}  mean={low.mean():.2f}')
print(f'\nt = {t_stat:.4f}   p (one-tailed) = {p_one:.4f}')
msg = 'REJECT H₀ — Significant difference' if p_one < 0.05 else 'FAIL TO REJECT H₀'
print(f'\nVerdict: {msg}')

fig, ax = plt.subplots(figsize=(8, 5))
parts = ax.violinplot([high, low], positions=[1,2], showmedians=True, showmeans=True)
for pc, col in zip(parts['bodies'], [COLORS[1], COLORS[3]]):
    pc.set_facecolor(col); pc.set_alpha(0.65)
ax.set_xticks([1,2])
ax.set_xticklabels([f'High attendance\n≥{threshold}%', f'Low attendance\n<{threshold}%'])
ax.set(ylabel='Final Exam %', title=f'Attendance vs Exam Score  |  p = {p_one:.4f}')
ax.grid(True)
plt.tight_layout()
plt.show()

---
## 6 · ✏️ Personal Grade Tracker — Edit and Run!

In [ ]:
# ── ✏️  EDIT YOUR MODULES & SCORES HERE ─────────────────────────────────
grades = {
    'Mathematics':      {'score': 72, 'credits': 20},
    'Chemistry':        {'score': 68, 'credits': 20},
    'Programming':      {'score': 81, 'credits': 15},
    'Lab Work':         {'score': 74, 'credits': 15},
    'Report Writing':   {'score': 65, 'credits': 10},
    'Research Project': {'score': 78, 'credits': 20},
}
# ─────────────────────────────────────────────────────────────────────────

gdf = pd.DataFrame(grades).T.reset_index()
gdf.columns = ['Module', 'Score', 'Credits']
gdf[['Score','Credits']] = gdf[['Score','Credits']].astype(float)
weighted_avg = (gdf['Score'] * gdf['Credits']).sum() / gdf['Credits'].sum()
classification = grade_band(weighted_avg)

print(gdf[['Module','Score','Credits']].to_string(index=False))
print(f'\nWeighted Average:  {weighted_avg:.2f}%')
print(f'UK Classification: {classification}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('My Grade Summary', fontsize=14, fontweight='bold', color='#94e2d5')

bars = axes[0].barh(gdf['Module'], gdf['Score'], color=COLORS[:len(gdf)], edgecolor='#0f1117')
axes[0].axvline(weighted_avg, color='white', lw=1.5, linestyle='--', label=f'Avg {weighted_avg:.1f}%')
axes[0].axvline(70, color='#a6e3a1', lw=1, linestyle=':', label='First (70)')
axes[0].set(xlabel='Score %', title='Module Scores', xlim=(0, 105))
axes[0].legend(fontsize=9); axes[0].grid(True, axis='x')
for bar, s in zip(bars, gdf['Score']):
    axes[0].text(s+0.5, bar.get_y()+bar.get_height()/2, f'{s:.0f}', va='center', fontsize=9)

axes[1].pie(gdf['Credits'], labels=gdf['Module'], colors=COLORS[:len(gdf)],
            autopct='%1.0f%%', startangle=90, pctdistance=0.82,
            wedgeprops={'edgecolor':'#0f1117','linewidth':1.5})
axes[1].set_title('Credit Weighting')
plt.tight_layout()
plt.show()

---
## ✅ Summary Table

| Section | Technique | Key Library |
|---------|-----------|-------------|
| Descriptive stats | mean, std, IQR | `pandas` |
| Distributions | Histograms + σ bands | `matplotlib` |
| Correlation | Pearson r heatmap | `pandas.corr` |
| Prediction | Multiple linear regression | `sklearn` |
| Hypothesis test | Independent t-test | `scipy.stats` |
| Grade tracker | Weighted average | `pandas` |

> **Tip:** Load real data with `df = pd.read_csv('my_data.csv')` and replace the synthetic dataset.